In [9]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

file_path = "data/eda/diamonds.csv"
df = pd.read_csv(file_path, dtype=str)

In [10]:
num_cols = ["carat", "depth", "table", "x", "y", "z", "price"]
cat_cols = ["cut", "color", "clarity"]

df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

# Apply this function because it does not generate data leakage
df["price"] = np.log1p(df["price"])

# Simple linear regression

We are going to predict the price of a diamond based on its carat measurement.

In [6]:
X_simple = df[["carat"]] 
y_simple = df["price"]

boxcox_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("boxcox", PowerTransformer(method="box-cox", standardize=True)),  # Does not need standard scaler
])

preprocess = ColumnTransformer([
    ("box-cox", boxcox_pipeline, ["carat"]),
])

model_simple = Pipeline([
    ("preprocess", preprocess),
    ("regression", LinearRegression()),
])

X_train_simple, X_test_simple, y_train_simple, y_test_simple = train_test_split(X_simple, y_simple, test_size=0.25, random_state=777)

model_simple.fit(X_train_simple, y_train_simple) # Here we fit and transform the information
y_pred = model_simple.predict(X_test_simple) # Here we only trasnform and predict

# Metrics
mae_simple = mean_absolute_error(y_test_simple, y_pred)
mse_simple = mean_squared_error(y_test_simple, y_pred)
r2_simple = r2_score(y_test_simple, y_pred)

print(f"MAE:  {mae_simple:.2f}")
print(f"MSE:  {mse_simple:.2f}")
print(f"R²:   {r2_simple:.4f}")

MAE:  0.21
MSE:  0.07
R²:   0.9320


# Two column linear regression

Lest try to use two columns. One numeric and one categoric.

In [8]:
X_two = df[["carat", "cut"]]  
y_two = df["price"]

boxcox_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("boxcox", PowerTransformer(method="box-cox", standardize=True)),  # Does not need standard scaler
])

catecorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder()),
])

preprocess = ColumnTransformer([
    ("box-cox", boxcox_pipeline, ["carat"]),
    ("cat", catecorical_pipeline, ["cut"]),
])

model_two = Pipeline([
    ("preprocess", preprocess),
    ("regression", LinearRegression()),
])

X_train_two, X_test_two, y_train_two, y_test_two = train_test_split(X_two, y_two, test_size=0.25, random_state=777)

model_two.fit(X_train_two, y_train_two) # Here we fit and transform the information
y_pred = model_two.predict(X_test_two) # Here we only trasnform and predict

# Metrics
mae = mean_absolute_error(y_test_two, y_pred)
mse = mean_squared_error(y_test_two, y_pred)
r2 = r2_score(y_test_two, y_pred)

print(f"MAE:  {mae:.2f}")
print(f"MSE:  {mse:.2f}")
print(f"R²:   {r2:.4f}")

MAE:  0.20
MSE:  0.07
R²:   0.9362


Lets add more columns so our model can be better!

In [11]:
stardard_cols = ["table"]
robust_cols = ["depth"]
boxcox_cols = ["carat"]
cat_cols = ["cut", "color", "clarity"]

X = df[stardard_cols + robust_cols + boxcox_cols + cat_cols]  
y = df["price"]

standard_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
])

robust_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
])

boxcox_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("boxcox", PowerTransformer(method="box-cox", standardize=True)),
])

catecorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder()),
])

preprocess = ColumnTransformer([
    ("standard", standard_pipeline, stardard_cols),
    ("robust", robust_pipeline, robust_cols),
    ("box-cox", boxcox_pipeline, boxcox_cols),
    ("cat", catecorical_pipeline, cat_cols),
])

model = Pipeline([
    ("preprocess", preprocess),
    ("regression", LinearRegression()),
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=777)

model.fit(X_train, y_train) # Here we fit and transform the information
y_pred = model.predict(X_test) # Here we only trasnform and predict

# Metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.2f}")
print(f"MSE:  {mse:.2f}")
print(f"R²:   {r2:.4f}")

MAE:  0.11
MSE:  0.02
R²:   0.9806
